# Phase 4 CJPE — Knowledge Distillation into a Compressed Phase-1 Student

**Teacher:** Phase 4 Experiment 1, `exp1_v2_encoder_full_ft_model.pt` (Stage A weights, trained on train only).  
**Student initialization:** the Phase 1 encoder–decoder checkpoint saved as `best_model_encdec.pt`. The student keeps four pretrained encoder blocks (`[0, 2, 3, 5]`), truncates the original byte-BPE vocabulary to **24,361 tokens**, drops the decoder/LM projection, and adds a lightweight document-classification head.

The student is designed for the same binary CJPE classification task. It learns from both gold labels and teacher soft targets:

`loss = (1 - alpha) * hard-label cross-entropy + alpha * T^2 * KL(student || teacher)`

The notebook uses the original A1 byte-BPE tokenizer, lowercases and compacts text as Experiment 1 did, caches teacher logits, validates on `dev.json`, and writes a Phase 4 submission. It asserts at runtime that the complete student has fewer than 50 million parameters.

**Before running:** use a Colab GPU, place `best_model_encdec.pt`, `legal_bpe_a1_50k_final.json`, and the Phase 4 Experiment 1 output directory in Google Drive. The source Experiment 1 output directory must contain its `token_cache` NPZ files. Set a Colab Secret named `HF_TOKEN` if the dataset repository needs authentication; the notebook never embeds a token.

## 1. Mount Drive, resolve checkpoints and output paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, os, json, math, gc, random, time, re, heapq, shutil, getpass, multiprocessing as mp
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

DRIVE_ROOT = Path('/content/drive/MyDrive')
TEACHER_DIR = DRIVE_ROOT / 'phase4_exp1_v2_encoder_full_ft'
TEACHER_CKPT_HINT = TEACHER_DIR / 'exp1_v2_encoder_full_ft_model.pt'
ENCDEC_CKPT_HINT = DRIVE_ROOT / 'best_model_encdec.pt'
TOKENIZER_HINT = DRIVE_ROOT / 'legal_bpe_a1_50k_final.json'
OUT_DIR = DRIVE_ROOT / 'phase4_kd_encdec_student'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def find_in_drive(hint, name):
    hint = Path(hint)
    if hint.is_file():
        return hint
    hits = [Path(p) for p in glob.glob(str(DRIVE_ROOT / '**' / name), recursive=True) if Path(p).is_file()]
    if not hits:
        raise FileNotFoundError(f'{name} was not found under {DRIVE_ROOT}. Update the path near the top of this cell.')
    # Prefer the path that is semantically closest to the hint, then the shallowest path.
    hits.sort(key=lambda p: (len(p.parts), str(p)))
    return hits[0]

TEACHER_CKPT_PATH = find_in_drive(TEACHER_CKPT_HINT, 'exp1_v2_encoder_full_ft_model.pt')
ENCDEC_CKPT_PATH = find_in_drive(ENCDEC_CKPT_HINT, 'best_model_encdec.pt')
TOKENIZER_PATH = find_in_drive(TOKENIZER_HINT, 'legal_bpe_a1_50k_final.json')
TEACHER_CACHE_DIR = TEACHER_DIR / 'token_cache'

print('Teacher checkpoint:', TEACHER_CKPT_PATH)
print('Phase-1 encoder-decoder checkpoint:', ENCDEC_CKPT_PATH)
print('Tokenizer:', TOKENIZER_PATH)
print('Teacher token cache:', TEACHER_CACHE_DIR)
print('Output directory:', OUT_DIR)

Mounted at /content/drive
Teacher checkpoint: /content/drive/MyDrive/phase4_exp1_v2_encoder_full_ft/exp1_v2_encoder_full_ft_model.pt
Phase-1 encoder-decoder checkpoint: /content/drive/MyDrive/best_model_encdec.pt
Tokenizer: /content/drive/MyDrive/legal_bpe_a1_50k_final.json
Teacher token cache: /content/drive/MyDrive/phase4_exp1_v2_encoder_full_ft/token_cache
Output directory: /content/drive/MyDrive/phase4_kd_encdec_student


## 2. Runtime, reproducibility and experiment settings

## Runtime scope

This notebook trains the student for **at most 4 validation epochs** and selects the best checkpoint using the development split. It does **not** run a second/final training stage on train+dev. After validation, it still runs inference on the test set to write the submission file; inference is not training. The existing 15-GB GPU-memory ceiling is respected: per-step batch size remains 4 and gradient accumulation remains 2.


In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('A Colab GPU is strongly recommended for the 16-layer teacher inference and student training. Select Runtime → Change runtime type → GPU.')

device = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
USE_SCALER = AMP_DTYPE == torch.float16

# Teacher preprocessing follows Phase 4 Experiment 1.
SPLITS = ['train', 'validation', 'test']
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPACE_ID = 36
TEACHER_K = 8
TEACHER_BODY = 511
TAIL_TOK_TEACHER = 4096
TAIL_CH = 28000

# Compressed student design. One token is the original byte-BPE space symbol,
# so removing it at cache time matches the compact Phase-4 representation.
REDUCED_VOCAB_SIZE = 24361
STUDENT_K = 4
STUDENT_BODY = 703
STUDENT_TAIL_TOK = STUDENT_K * STUDENT_BODY
SELECTED_ENCODER_LAYERS = [0, 2, 3, 5]
STUDENT_LSTM_HIDDEN = 192
DISTILL_TEMPERATURE = 2.0
ALPHA_KD = 0.50

BATCH_DOCS = 4
GRAD_ACCUM_STEPS = 2
EPOCHS = 4  # validation training only; no final refit stage
PATIENCE = 2
MIN_DELTA_AUC = 0.0005
PRETRAINED_LR = 1e-5
HEAD_LR = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.08
MAX_GRAD_NORM = 1.0
NUM_TOKEN_WORKERS = max(1, min(4, os.cpu_count() or 2))

print('GPU:', GPU_NAME, f'({GPU_GB:.1f} GB)')
print('AMP dtype:', AMP_DTYPE, '| GradScaler:', USE_SCALER)
print('Student vocabulary:', f'{REDUCED_VOCAB_SIZE:,}')
print('Student selected Phase-1 encoder layers:', SELECTED_ENCODER_LAYERS)
print('Student chunk context:', STUDENT_BODY + 1, '| max chunks/doc:', STUDENT_K)

GPU: NVIDIA A100-SXM4-40GB (39.5 GB)
AMP dtype: torch.bfloat16 | GradScaler: False
Student vocabulary: 24,361
Student selected Phase-1 encoder layers: [0, 2, 3, 5]
Student chunk context: 704 | max chunks/doc: 4


## 3. Load the exact Phase-4 A1 BPE tokenizer

In [ ]:
N_SPECIAL, N_BYTES = 4, 256
BASE_VOCAB_SIZE = N_SPECIAL + N_BYTES
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

class A1Tokenizer:
    def __init__(self, itos, merges):
        self.itos = list(itos)
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos)
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}
        self._id_to_bytes = [b""] * N_SPECIAL + [bytes([b]) for b in range(256)]
        for l, r in self.merges:
            self._id_to_bytes.append(self._id_to_bytes[l] + self._id_to_bytes[r])
        assert len(self._id_to_bytes) == self.vocab_size
        self._cache = {}

    def _encode_word(self, word):                       # identical to Phase 1
        raw = word.encode("utf-8")
        seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks:
            return seq
        n = len(seq)
        prev = [-1] + list(range(0, n - 1))
        nxt = list(range(1, n)) + [-1]
        active = [True] * n
        heap = []
        for i in range(n - 1):
            rank = self._merge_ranks.get((seq[i], seq[i + 1]))
            if rank is not None:
                heapq.heappush(heap, (rank, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]:
                continue
            if nxt[left] != right or prev[right] != left:
                continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank:
                continue
            seq[left] = BASE_VOCAB_SIZE + rank
            active[right] = False
            right_next = nxt[right]
            nxt[left] = right_next
            if right_next != -1:
                prev[right_next] = left
            left_prev = prev[left]
            if left_prev != -1:
                new_rank = self._merge_ranks.get((seq[left_prev], seq[left]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left_prev, left))
            if right_next != -1:
                new_rank = self._merge_ranks.get((seq[left], seq[right_next]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left, right_next))
        out, i = [], 0
        while i != -1:
            if active[i]:
                out.append(seq[i])
            i = nxt[i]
        return out

    def encode_normalized(self, text):
        # text: single-space separated, no leading/trailing space
        out, cache, first = [], self._cache, True
        for w in text.split(" "):
            if not first:
                out.append(N_SPECIAL + 32)
            first = False
            ids = cache.get(w)
            if ids is None:
                ids = self._encode_word(w)
                if len(cache) > 1_500_000:
                    cache.clear()
                cache[w] = ids
            out.extend(ids)
        return out

    def decode(self, ids):
        buf, chunks = bytearray(), []
        for i in ids:
            i = int(i)
            if i < N_SPECIAL:
                chunks.append(bytes(buf).decode("utf-8", errors="replace")); buf.clear()
                chunks.append(SPECIAL_TOKENS[i])
            else:
                buf.extend(self._id_to_bytes[i])
        chunks.append(bytes(buf).decode("utf-8", errors="replace"))
        return "".join(chunks)

with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    _payload = json.load(f)
TOK = A1Tokenizer(_payload["itos"], _payload["merges"])
assert TOK.vocab_size == 50000 and TOK.itos[:4] == SPECIAL_TOKENS
assert N_SPECIAL + 32 == SPACE_ID

_t = "The Supreme Court held that the appeal was dismissed. भारत के संविधान के अनुसार।"
_ids = TOK.encode_normalized(_t)
assert TOK.decode(_ids) == _t
print("tokenizer ready | vocab", TOK.vocab_size, "| round-trip OK | space token id", SPACE_ID)

tokenizer ready | vocab 50000 | round-trip OK | space token id 36


## 4. Restore the Phase-4 full-fine-tuned teacher architecture

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 50000
    d_model: int = 768
    n_layers: int = 16
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_seq_len: int = 512
    rope_base: float = 10000.0
    rms_norm_eps: float = 1e-6
    dropout: float = 0.0

def build_rope_cache(max_seq_len, d_head, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    ang = torch.cat([freqs, freqs], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    cos = cos.to(device=x.device, dtype=x.dtype)[None, None]
    sin = sin.to(device=x.device, dtype=x.dtype)[None, None]
    return x * cos + rotate_half(x) * sin

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class BidirectionalGQAAttention(nn.Module):
    # Phase-1 attention + an optional key-padding mask (Phase 1 never needed one: all windows were full)
    def __init__(self, c):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = c.d_model, c.n_heads, c.n_kv_heads
        self.head_dim = c.d_model // c.n_heads
        self.n_rep = c.n_heads // c.n_kv_heads
        self.max_seq_len = c.max_seq_len
        self.q_proj = nn.Linear(c.d_model, c.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(c.d_model, c.d_model, bias=False)
        cos, sin = build_rope_cache(c.max_seq_len, self.head_dim, c.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _split(self, x, h):
        B, L, _ = x.shape
        return x.reshape(B, L, h, self.head_dim).transpose(1, 2)

    def forward(self, x, key_mask=None):
        B, L, _ = x.shape
        q = self._split(self.q_proj(x), self.n_heads)
        k = self._split(self.k_proj(x), self.n_kv_heads)
        v = self._split(self.v_proj(x), self.n_kv_heads)
        q = apply_rope(q, self.rope_cos[:L], self.rope_sin[:L])
        k = apply_rope(k, self.rope_cos[:L], self.rope_sin[:L])
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        mask = None if key_mask is None else key_mask[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        return self.out_proj(a.transpose(1, 2).contiguous().view(B, L, self.d_model))


class EncoderBlock(nn.Module):
    def __init__(self, c, p_drop=0.0):
        super().__init__()
        self.attn_norm = RMSNorm(c.d_model, c.rms_norm_eps)
        self.attn = BidirectionalGQAAttention(c)
        self.ffn_norm = RMSNorm(c.d_model, c.rms_norm_eps)
        self.mlp = SwiGLU(c.d_model, c.d_ff)
        self.drop = nn.Dropout(p_drop)          # no parameters -> checkpoint-compatible
    def forward(self, x, key_mask=None):
        x = x + self.drop(self.attn(self.attn_norm(x), key_mask))
        return x + self.drop(self.mlp(self.ffn_norm(x)))

class LegalEncoder(nn.Module):
    # parameter names match the Phase-1 checkpoint (minus the tied mlm_head)
    def __init__(self, c, grad_ckpt=False, p_drop=0.0):
        super().__init__()
        self.grad_ckpt = grad_ckpt
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.layers = nn.ModuleList([EncoderBlock(c, p_drop) for _ in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)
    def forward(self, ids, key_mask=None):
        if key_mask is not None and bool(key_mask.all()):
            key_mask = None
        x = self.token_embedding(ids)
        for layer in self.layers:
            if self.grad_ckpt and self.training:
                x = checkpoint(layer, x, key_mask, use_reentrant=False)
            else:
                x = layer(x, key_mask)
        return self.final_norm(x)

class LegalDocClassifier(nn.Module):
    # window = mean-pooled encoder output; document = attention over windows (+ distance-from-end embedding),
    # concatenated with the last window's vector, then a linear classifier.
    def __init__(self, c, max_chunks=None, resid_drop=0.0, head_drop=0.1, grad_ckpt=False):
        if max_chunks is None:
            max_chunks = TEACHER_K
        super().__init__()
        d = c.d_model
        self.max_chunks = max_chunks
        self.encoder = LegalEncoder(c, grad_ckpt, resid_drop)
        self.chunk_pos = nn.Embedding(max_chunks, d)
        nn.init.normal_(self.chunk_pos.weight, std=0.02)
        self.chunk_attn = nn.Sequential(nn.Linear(d, 256), nn.Tanh(), nn.Linear(256, 1, bias=False))
        self.dropout = nn.Dropout(head_drop)
        self.classifier = nn.Linear(2 * d, 2)
        nn.init.normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)
        self.last_chunk_attn, self.last_chunk_mask = None, None

    def forward(self, batch):
        am = batch["attn_mask"]
        h = self.encoder(batch["input_ids"], am).float()
        m = am.unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1).clamp(min=1)                       # [N, d]
        B, Kc, d = batch["n_docs"], self.max_chunks, v.shape[-1]
        di, sl = batch["doc_index"], batch["slot"]
        chunks = v.new_zeros(B, Kc, d).index_put((di, sl), v)
        cm = torch.zeros(B, Kc, dtype=torch.bool, device=v.device)
        cm[di, sl] = True
        pe = self.chunk_pos(torch.arange(Kc - 1, -1, -1, device=v.device))   # slot s -> distance Kc-1-s from the end
        sc = self.chunk_attn(chunks + pe.unsqueeze(0)).squeeze(-1).float().masked_fill(~cm, float("-inf"))
        w = torch.softmax(sc, dim=-1)
        agg = (w.unsqueeze(-1) * chunks).sum(1)
        doc = torch.cat([agg, chunks[:, Kc - 1]], dim=-1)               # last chunk always present
        self.last_chunk_attn, self.last_chunk_mask = w.detach(), cm
        return self.classifier(self.dropout(doc))

In [ ]:
TEACHER_CHECKPOINT = torch.load(TEACHER_CKPT_PATH, map_location='cpu', weights_only=False)
if 'model_state_dict_stageA' not in TEACHER_CHECKPOINT:
    raise KeyError('The teacher checkpoint does not contain model_state_dict_stageA. Use the checkpoint saved by phase4_exp1_v2_encoder_full_ft_colab (1)_1.ipynb.')

teacher_cfg = ModelConfig(**TEACHER_CHECKPOINT['encoder_config'])
TEACHER_K = int(TEACHER_CHECKPOINT.get('K', TEACHER_K))
teacher_model = LegalDocClassifier(
    teacher_cfg,
    max_chunks=TEACHER_K,
    resid_drop=float(TEACHER_CHECKPOINT.get('resid_dropout', 0.1)),
    head_drop=float(TEACHER_CHECKPOINT.get('head_dropout', 0.1)),
    grad_ckpt=False,
).to(device)
missing, unexpected = teacher_model.load_state_dict(TEACHER_CHECKPOINT['model_state_dict_stageA'], strict=True)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad_(False)

print('Teacher weights loaded: Stage A (train-only weights; dev was only used for model selection).')
print('Teacher parameters:', f'{sum(p.numel() for p in teacher_model.parameters()):,}')
print('Teacher config:', asdict(teacher_cfg))
del missing, unexpected
gc.collect()

Teacher weights loaded: Stage A (train-only weights; dev was only used for model selection).
Teacher parameters: 139,294,978
Teacher config: {'vocab_size': 50000, 'd_model': 768, 'n_layers': 16, 'n_heads': 12, 'n_kv_heads': 4, 'd_ff': 2048, 'max_seq_len': 512, 'rope_base': 10000.0, 'rms_norm_eps': 1e-06, 'dropout': 0.0}


10

## 5. Load Experiment 1's token cache

The teacher cache contains the unique case IDs, compact token tails, offsets, and gold labels. We reuse it rather than retokenizing the teacher inputs. Validation and test labels are never fabricated; the test cache is expected to have no `labels` field.

In [ ]:
TEACHER_CACHE_SIG = f'v2_lower_compact_t{TAIL_TOK_TEACHER}_ch{TAIL_CH}'
def teacher_cache_path(split):
    return TEACHER_CACHE_DIR / f'{split}_{TEACHER_CACHE_SIG}.npz'

missing_caches = [str(teacher_cache_path(s)) for s in SPLITS if not teacher_cache_path(s).is_file()]
if missing_caches:
    raise FileNotFoundError(
        'Missing Experiment 1 token cache files. Run the token-cache cell in the Phase 4 Experiment 1 notebook first. Missing: ' + ', '.join(missing_caches)
    )

T_D = {}
for split in SPLITS:
    z = np.load(teacher_cache_path(split), allow_pickle=False)
    T_D[split] = {k: z[k] for k in z.files}
    T_D[split]['n'] = len(T_D[split]['ids'])
    assert len(set(T_D[split]['ids'].tolist())) == T_D[split]['n']
    assert ('labels' in T_D[split]) == (split != 'test')
    assert len(T_D[split]['tail_off']) == T_D[split]['n'] + 1


def teacher_tail(split, i):
    d = T_D[split]
    return d['tail_flat'][d['tail_off'][i]:d['tail_off'][i + 1]]

print('Teacher cache loaded:')
for split in SPLITS:
    d = T_D[split]
    label_note = f" | positive rate={d['labels'].mean():.3f}" if 'labels' in d else ' | test labels not present'
    print(f'  {split:<12} {d["n"]:>7,} unique cases{label_note}')
assert T_D['test']['n'] == 1517

Teacher cache loaded:
  train         32,305 unique cases | positive rate=0.414
  validation       994 unique cases | positive rate=0.500
  test           1,517 unique cases | test labels not present


## 6. Build/load the 24,361-token student cache

In [ ]:
STUDENT_CACHE_DIR = OUT_DIR / 'student_token_cache'
STUDENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
STUDENT_CACHE_SIG = f'v{REDUCED_VOCAB_SIZE}_k{STUDENT_K}_body{STUDENT_BODY}_ch{TAIL_CH}'
def student_cache_path(split):
    return STUDENT_CACHE_DIR / f'{split}_{STUDENT_CACHE_SIG}.npz'

# Preserve original token IDs for the first 24,361 entries. Since A1-BPE merge IDs
# are assigned in merge order, keeping the first V-260 merges gives a valid prefix tokenizer.
BASE_VOCAB_SIZE = 4 + 256
assert TOK.vocab_size == 50000
student_merges = TOK.merges[:REDUCED_VOCAB_SIZE - BASE_VOCAB_SIZE]
STUDENT_TOK = A1Tokenizer(TOK.itos[:REDUCED_VOCAB_SIZE], student_merges)
assert STUDENT_TOK.vocab_size == REDUCED_VOCAB_SIZE
assert SPACE_ID == 36
print('Student tokenizer ready | vocab:', STUDENT_TOK.vocab_size, '| merges:', len(student_merges))

REPO_ID = 'Exploration-Lab/CS-781-Capstone'
REMOTE_FILES = {
    'train': 'Phase_4_CJPE/Train/train.json',
    'validation': 'Phase_4_CJPE/Train/dev.json',
    'test': 'Phase_4_CJPE/Test/test.json',
}
DATA_DIR = Path('/content/cjpe_kd_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def get_hf_token():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        pass
    if not token:
        token = os.environ.get('HF_TOKEN')
    if not token:
        token = getpass.getpass('Hugging Face read token (input hidden; not saved to the notebook): ').strip()
    return token or None

STUDENT_CACHE_READY = all(student_cache_path(s).exists() for s in SPLITS)
if not STUDENT_CACHE_READY:
    from huggingface_hub import hf_hub_download
    HF_TOKEN = get_hf_token()

    def load_json_records(path):
        with open(path, 'r', encoding='utf-8') as f:
            first = f.read(1)
            f.seek(0)
            if first == '[':
                return json.load(f)
            return [json.loads(line) for line in f if line.strip()]

    def fetch_raw_split(split):
        local = hf_hub_download(
            repo_id=REPO_ID,
            filename=REMOTE_FILES[split],
            repo_type='dataset',
            token=HF_TOKEN,
            local_dir=str(DATA_DIR),
        )
        raw = load_json_records(local)
        seen, ids, texts, labels = set(), [], [], []
        for row in raw:
            cid = str(row['id'])
            if cid in seen:
                continue
            seen.add(cid)
            ids.append(cid)
            texts.append(row.get('text') or '')
            if split != 'test':
                labels.append(int(row['label']))
        if ids != [str(x) for x in T_D[split]['ids'].tolist()]:
            raise ValueError(f'{split}: raw dataset case ordering/IDs do not match the Experiment 1 teacher cache. Do not train until these align.')
        if split != 'test' and not np.array_equal(np.asarray(labels, dtype=np.int64), T_D[split]['labels'].astype(np.int64)):
            raise ValueError(f'{split}: raw labels do not match the Experiment 1 teacher cache.')
        return ids, texts, labels

    _TOK_TEXTS = []
    def _student_tok_job(i):
        raw = _TOK_TEXTS[i]
        s = ' '.join(raw.lower().split()) or '.'
        if len(s) > TAIL_CH:
            s = s[-TAIL_CH:]
            cut = s.find(' ')
            s = s[cut + 1:] if cut >= 0 else s
        ids = np.asarray(STUDENT_TOK.encode_normalized(s), dtype=np.int64)
        ids = ids[ids != SPACE_ID]
        ids = ids[-STUDENT_TAIL_TOK:]
        if ids.size and int(ids.max()) >= REDUCED_VOCAB_SIZE:
            raise ValueError('Reduced tokenizer emitted an out-of-vocabulary ID.')
        return ids.astype(np.uint16)

    def tokenize_student_texts(texts):
        global _TOK_TEXTS
        _TOK_TEXTS = texts
        ctx = mp.get_context('fork')
        result = []
        with ctx.Pool(NUM_TOKEN_WORKERS) as pool:
            for i, ids in enumerate(pool.imap(_student_tok_job, range(len(texts)), chunksize=8), 1):
                result.append(ids)
                if i % max(1, len(texts)//10) == 0 or i == len(texts):
                    print(f'  tokenized {i:,}/{len(texts):,}')
        _TOK_TEXTS = []
        return result

    for split in SPLITS:
        outpath = student_cache_path(split)
        if outpath.exists():
            print(f'{split}: using existing student cache')
            continue
        print(f'\n[{split}] downloading text and building reduced-vocabulary cache')
        ids, texts, labels = fetch_raw_split(split)
        result = tokenize_student_texts(texts)
        offsets = np.zeros(len(result) + 1, dtype=np.int64)
        offsets[1:] = np.cumsum([len(x) for x in result])
        flat = np.concatenate(result).astype(np.uint16) if result else np.empty(0, dtype=np.uint16)
        payload = {
            'ids': np.asarray(ids),
            'tail_flat': flat,
            'tail_off': offsets,
        }
        if split != 'test':
            payload['labels'] = np.asarray(labels, dtype=np.int64)
        np.savez_compressed(outpath, **payload)
        del result, flat, offsets, payload, texts
        gc.collect()
        print('  saved:', outpath)

S_D = {}
for split in SPLITS:
    z = np.load(student_cache_path(split), allow_pickle=False)
    S_D[split] = {k: z[k] for k in z.files}
    S_D[split]['n'] = len(S_D[split]['ids'])
    if not np.array_equal(S_D[split]['ids'].astype(str), T_D[split]['ids'].astype(str)):
        raise ValueError(f'{split}: reduced-vocabulary cache IDs do not align with the teacher cache.')
    assert int(S_D[split]['tail_flat'].max(initial=0)) < REDUCED_VOCAB_SIZE
    assert ('labels' in S_D[split]) == (split != 'test')


def student_tail(split, i):
    d = S_D[split]
    return d['tail_flat'][d['tail_off'][i]:d['tail_off'][i + 1]]

print('\nStudent cache ready:')
for split in SPLITS:
    d = S_D[split]
    print(f'  {split:<12} {d["n"]:>7,} unique cases | total compact tokens={len(d["tail_flat"]):,}')

Student tokenizer ready | vocab: 24361 | merges: 24101
Hugging Face read token (input hidden; not saved to the notebook): ··········

[train] downloading text and building reduced-vocabulary cache


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

  tokenized 3,230/32,305
  tokenized 6,460/32,305
  tokenized 9,690/32,305
  tokenized 12,920/32,305
  tokenized 16,150/32,305
  tokenized 19,380/32,305
  tokenized 22,610/32,305
  tokenized 25,840/32,305
  tokenized 29,070/32,305
  tokenized 32,300/32,305
  tokenized 32,305/32,305
  saved: /content/drive/MyDrive/phase4_kd_encdec_student/student_token_cache/train_v24361_k4_body703_ch28000.npz

[validation] downloading text and building reduced-vocabulary cache


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

  tokenized 99/994
  tokenized 198/994
  tokenized 297/994
  tokenized 396/994
  tokenized 495/994
  tokenized 594/994
  tokenized 693/994
  tokenized 792/994
  tokenized 891/994
  tokenized 990/994
  tokenized 994/994
  saved: /content/drive/MyDrive/phase4_kd_encdec_student/student_token_cache/validation_v24361_k4_body703_ch28000.npz

[test] downloading text and building reduced-vocabulary cache


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

  tokenized 151/1,517
  tokenized 302/1,517
  tokenized 453/1,517
  tokenized 604/1,517
  tokenized 755/1,517
  tokenized 906/1,517
  tokenized 1,057/1,517
  tokenized 1,208/1,517
  tokenized 1,359/1,517
  tokenized 1,510/1,517
  tokenized 1,517/1,517
  saved: /content/drive/MyDrive/phase4_kd_encdec_student/student_token_cache/test_v24361_k4_body703_ch28000.npz

Student cache ready:
  train         32,305 unique cases | total compact tokens=62,978,681
  validation       994 unique cases | total compact tokens=2,400,219
  test           1,517 unique cases | total compact tokens=3,689,397


## 7. Cache the teacher's document-level logits

The teacher's **Stage A** parameters are used here because that stage was trained on the training split only. Stage B was retrained with repeated dev documents and would make teacher predictions on dev unsuitable for unbiased student model selection.

In [ ]:
def teacher_n_windows(split, i):
    L = int(T_D[split]['tail_off'][i + 1] - T_D[split]['tail_off'][i])
    return min(TEACHER_K, max(1, (L + TEACHER_BODY - 1) // TEACHER_BODY))

def build_teacher_chunks(split, i):
    tail = teacher_tail(split, i).astype(np.int64)
    body, end = [], len(tail)
    while end > 0 and len(body) < TEACHER_K:
        start = max(0, end - TEACHER_BODY)
        body.append(tail[start:end])
        end = start
    body = body[::-1] if body else [tail]
    n = len(body)
    return [(TEACHER_K - n + j, np.concatenate(([BOS_ID], chunk)).astype(np.int64)) for j, chunk in enumerate(body)]

def pack_doc_items(items, max_windows, window_counter):
    groups, cur, ncur = [], [], 0
    for item in items:
        nw = window_counter(*item)
        if cur and ncur + nw > max_windows:
            groups.append(cur)
            cur, ncur = [], 0
        cur.append(item); ncur += nw
    if cur:
        groups.append(cur)
    return groups

def collate_teacher(items):
    seqs, didx, slots = [], [], []
    for b, (split, i) in enumerate(items):
        for slot, ids in build_teacher_chunks(split, i):
            seqs.append(ids); didx.append(b); slots.append(slot)
    L = max(map(len, seqs))
    input_ids = np.full((len(seqs), L), PAD_ID, dtype=np.int64)
    attn_mask = np.zeros((len(seqs), L), dtype=bool)
    for j, ids in enumerate(seqs):
        input_ids[j, :len(ids)] = ids
        attn_mask[j, :len(ids)] = True
    return {
        'input_ids': torch.from_numpy(input_ids),
        'attn_mask': torch.from_numpy(attn_mask),
        'doc_index': torch.tensor(didx, dtype=torch.long),
        'slot': torch.tensor(slots, dtype=torch.long),
        'n_docs': len(items),
    }

def compute_teacher_logits(split, max_windows=16):
    items = [(split, i) for i in range(T_D[split]['n'])]
    groups = pack_doc_items(items, max_windows, teacher_n_windows)
    outputs = np.zeros((len(items), 2), dtype=np.float32)
    teacher_model.eval()
    with torch.inference_mode():
        for step, group in enumerate(groups, 1):
            batch = collate_teacher(group)
            batch = {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = teacher_model(batch)
            logits = logits.float().cpu().numpy()
            for j, (_, doc_i) in enumerate(group):
                outputs[doc_i] = logits[j]
            if step == 1 or step % 250 == 0 or step == len(groups):
                print(f'[{split}] teacher batch {step:,}/{len(groups):,}')
    return outputs

TEACHER_LOGITS_PATH = OUT_DIR / 'teacher_stageA_logits.npz'
TEACHER_META_PATH = OUT_DIR / 'teacher_stageA_logits_meta.json'
teacher_stat = TEACHER_CKPT_PATH.stat()
expected_meta = {
    'checkpoint': str(TEACHER_CKPT_PATH),
    'checkpoint_size': int(teacher_stat.st_size),
    'checkpoint_mtime': float(teacher_stat.st_mtime),
    'train_ids': len(T_D['train']['ids']),
    'validation_ids': len(T_D['validation']['ids']),
    'stage': 'model_state_dict_stageA',
}
TEACHER_LOGITS = None
if TEACHER_LOGITS_PATH.exists() and TEACHER_META_PATH.exists():
    try:
        saved_meta = json.load(open(TEACHER_META_PATH, 'r', encoding='utf-8'))
        if saved_meta == expected_meta:
            z = np.load(TEACHER_LOGITS_PATH, allow_pickle=False)
            if np.array_equal(z['train_ids'].astype(str), T_D['train']['ids'].astype(str)) and np.array_equal(z['validation_ids'].astype(str), T_D['validation']['ids'].astype(str)):
                TEACHER_LOGITS = {'train': z['train_logits'].astype(np.float32), 'validation': z['validation_logits'].astype(np.float32)}
                print('Loaded cached Stage A teacher logits:', TEACHER_LOGITS_PATH)
    except Exception as exc:
        print('Teacher-logit cache will be recomputed:', repr(exc))

if TEACHER_LOGITS is None:
    print('Computing teacher logits. This is a one-time pass and may take a while.')
    TEACHER_LOGITS = {
        'train': compute_teacher_logits('train'),
        'validation': compute_teacher_logits('validation'),
    }
    np.savez_compressed(
        TEACHER_LOGITS_PATH,
        train_ids=T_D['train']['ids'], validation_ids=T_D['validation']['ids'],
        train_logits=TEACHER_LOGITS['train'], validation_logits=TEACHER_LOGITS['validation'],
    )
    json.dump(expected_meta, open(TEACHER_META_PATH, 'w', encoding='utf-8'), indent=2)
    print('Saved teacher-logit cache:', TEACHER_LOGITS_PATH)

assert TEACHER_LOGITS['train'].shape == (T_D['train']['n'], 2)
assert TEACHER_LOGITS['validation'].shape == (T_D['validation']['n'], 2)
print('Teacher logit cache shapes:', {k: v.shape for k, v in TEACHER_LOGITS.items()})

Computing teacher logits. This is a one-time pass and may take a while.
[train] teacher batch 1/11,416
[train] teacher batch 250/11,416
[train] teacher batch 500/11,416
[train] teacher batch 750/11,416
[train] teacher batch 1,000/11,416
[train] teacher batch 1,250/11,416
[train] teacher batch 1,500/11,416
[train] teacher batch 1,750/11,416
[train] teacher batch 2,000/11,416
[train] teacher batch 2,250/11,416
[train] teacher batch 2,500/11,416
[train] teacher batch 2,750/11,416
[train] teacher batch 3,000/11,416
[train] teacher batch 3,250/11,416
[train] teacher batch 3,500/11,416
[train] teacher batch 3,750/11,416
[train] teacher batch 4,000/11,416
[train] teacher batch 4,250/11,416
[train] teacher batch 4,500/11,416
[train] teacher batch 4,750/11,416
[train] teacher batch 5,000/11,416
[train] teacher batch 5,250/11,416
[train] teacher batch 5,500/11,416
[train] teacher batch 5,750/11,416
[train] teacher batch 6,000/11,416
[train] teacher batch 6,250/11,416
[train] teacher batch 6,500/

## 8. Build the compressed Phase-1 encoder student

The vocabulary is reduced to the first 24,361 A1-BPE IDs, so the associated embedding rows keep their original token meanings. Four pretrained encoder blocks are retained and renumbered `[0, 1, 2, 3]` in the student. The decoder blocks and vocabulary-sized LM head are not instantiated. The task head consumes token representations from the retained encoder plus a two-layer BiLSTM, then aggregates up to four chunks into a single binary prediction per document.

In [ ]:
@dataclass
class StudentEncoderConfig:
    vocab_size: int = REDUCED_VOCAB_SIZE
    d_model: int = 768
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_encoder_len: int = STUDENT_BODY + 1
    rope_base: float = 10000.0
    rms_norm_eps: float = 1e-6
    dropout: float = 0.0


def student_rope_cache(max_seq_len, d_head, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()

def student_rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def student_apply_rope(x, cos, sin):
    cos = cos.to(device=x.device, dtype=x.dtype)[None, None]
    sin = sin.to(device=x.device, dtype=x.dtype)[None, None]
    return x * cos + student_rotate_half(x) * sin

def student_split_heads(x, n_heads, head_dim):
    b, t, _ = x.shape
    return x.reshape(b, t, n_heads, head_dim).transpose(1, 2)

def student_merge_heads(x):
    b, h, t, d = x.shape
    return x.transpose(1, 2).contiguous().reshape(b, t, h * d)

def student_grouped_attention(q, k, v, key_valid_mask=None):
    # Fused scaled-dot-product attention avoids materializing the large [batch, heads, T, T]
    # score tensor. Keep the original implementation as a compatibility fallback.
    if hasattr(F, 'scaled_dot_product_attention'):
        try:
            attn_mask = key_valid_mask[:, None, None, :] if key_valid_mask is not None else None
            # PyTorch 2.x supports native grouped-query attention with enable_gqa=True.
            return F.scaled_dot_product_attention(
                q, k, v, attn_mask=attn_mask, dropout_p=0.0, is_causal=False, enable_gqa=True
            )
        except (TypeError, RuntimeError, NotImplementedError):
            pass
    b, hq, tq, d = q.shape
    hkv = k.shape[1]
    n_rep = hq // hkv
    qg = q.reshape(b, hkv, n_rep, tq, d)
    kg, vg = k.unsqueeze(2), v.unsqueeze(2)
    scores = torch.matmul(qg, kg.transpose(-2, -1)) / math.sqrt(d)
    if key_valid_mask is not None:
        scores = scores.masked_fill(~key_valid_mask[:, None, None, None, :], torch.finfo(scores.dtype).min)
    weights = F.softmax(scores, dim=-1)
    out = torch.matmul(weights, vg).reshape(b, hq, tq, d)
    return out

class StudentEDEncoderSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.d_model = cfg.d_model
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        cos, sin = student_rope_cache(cfg.max_encoder_len, self.head_dim, cfg.rope_base)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def forward(self, x, key_valid_mask=None):
        b, t, _ = x.shape
        if t > self.rope_cos.shape[0]:
            raise ValueError('Student sequence exceeds max_encoder_len.')
        q = student_split_heads(self.q_proj(x), self.n_heads, self.head_dim)
        k = student_split_heads(self.k_proj(x), self.n_kv_heads, self.head_dim)
        v = student_split_heads(self.v_proj(x), self.n_kv_heads, self.head_dim)
        q = student_apply_rope(q, self.rope_cos[:t], self.rope_sin[:t])
        k = student_apply_rope(k, self.rope_cos[:t], self.rope_sin[:t])
        return self.out_proj(student_merge_heads(student_grouped_attention(q, k, v, key_valid_mask)))

class StudentEDEncoderBlock(nn.Module):
    # Parameter names/shapes intentionally match Phase 1 EncoderBlock.
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.self_attn = StudentEDEncoderSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.d_ff)
    def forward(self, x, key_valid_mask=None):
        x = x + self.self_attn(self.norm1(x), key_valid_mask=key_valid_mask)
        x = x + self.mlp(self.norm2(x))
        return x

class CompressedCJPEStudent(nn.Module):
    def __init__(self, cfg, n_chunks=STUDENT_K, lstm_hidden=STUDENT_LSTM_HIDDEN, dropout=0.1):
        super().__init__()
        self.cfg = cfg
        self.n_chunks = n_chunks
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder_blocks = nn.ModuleList([StudentEDEncoderBlock(cfg) for _ in SELECTED_ENCODER_LAYERS])
        self.final_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.lstm = nn.LSTM(
            input_size=cfg.d_model, hidden_size=lstm_hidden, num_layers=2,
            batch_first=True, bidirectional=True, dropout=dropout,
        )
        self.drop_in = nn.Dropout(dropout)
        self.drop_out = nn.Dropout(dropout)
        self.feature_dim = cfg.d_model + 2 * lstm_hidden
        self.chunk_pos = nn.Embedding(n_chunks, self.feature_dim)
        self.chunk_attn = nn.Sequential(
            nn.Linear(self.feature_dim, 256), nn.Tanh(), nn.Linear(256, 1, bias=False)
        )
        self.classifier = nn.Linear(2 * self.feature_dim, 2)

    def encode(self, ids, lengths):
        valid = torch.arange(ids.shape[1], device=ids.device).unsqueeze(0) < lengths.to(ids.device).unsqueeze(1)
        h = self.token_embedding(ids)
        for block in self.encoder_blocks:
            h = block(h, key_valid_mask=valid)
        return self.final_norm(h), valid

    def forward(self, batch):
        ids = batch['input_ids']
        lengths = batch['lengths']
        doc_index = batch['doc_index']
        slots = batch['slot']
        n_docs = int(batch['n_docs'])
        h, valid = self.encode(ids, lengths)
        packed = nn.utils.rnn.pack_padded_sequence(h, lengths.detach().cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=h.shape[1])
        features = self.drop_out(torch.cat([self.drop_in(h), out.to(h.dtype)], dim=-1))
        m = valid.unsqueeze(-1).to(features.dtype)
        chunk_vec = (features * m).sum(dim=1) / lengths.to(features.device, features.dtype).unsqueeze(-1).clamp(min=1)

        chunks = chunk_vec.new_zeros(n_docs, self.n_chunks, self.feature_dim)
        chunks.index_put_((doc_index, slots), chunk_vec)
        chunk_mask = torch.zeros(n_docs, self.n_chunks, dtype=torch.bool, device=chunk_vec.device)
        chunk_mask[doc_index, slots] = True
        pos_ids = torch.arange(self.n_chunks - 1, -1, -1, device=chunk_vec.device)
        scores = self.chunk_attn(chunks + self.chunk_pos(pos_ids).unsqueeze(0)).squeeze(-1)
        scores = scores.float().masked_fill(~chunk_mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1).to(chunks.dtype)
        pooled = (weights.unsqueeze(-1) * chunks).sum(dim=1)
        doc_vec = torch.cat([pooled, chunks[:, self.n_chunks - 1]], dim=-1)
        return self.classifier(self.drop_out(doc_vec))

student_cfg = StudentEncoderConfig()
student_model = CompressedCJPEStudent(student_cfg).to(device)

# Load the Phase-1 checkpoint on CPU; copy only token embedding rows and four encoder blocks.
encdec_ckpt = torch.load(ENCDEC_CKPT_PATH, map_location='cpu', weights_only=False)
if 'model_state_dict' not in encdec_ckpt:
    raise KeyError('best_model_encdec.pt must contain model_state_dict.')
source_state = encdec_ckpt['model_state_dict']
source_cfg = encdec_ckpt.get('model_config', {})
if source_cfg:
    if int(source_cfg.get('vocab_size', 50000)) != 50000 or int(source_cfg.get('d_model', 768)) != 768:
        raise ValueError(f'Unexpected Phase-1 encoder-decoder config: {source_cfg}')

with torch.no_grad():
    student_model.token_embedding.weight.copy_(source_state['token_embedding.weight'][:REDUCED_VOCAB_SIZE])
    student_model.final_norm.weight.copy_(source_state['final_norm.weight'])
    for new_i, old_i in enumerate(SELECTED_ENCODER_LAYERS):
        dst = dict(student_model.encoder_blocks[new_i].named_parameters())
        for suffix, dst_param in dst.items():
            src_key = f'encoder_blocks.{old_i}.{suffix}'
            if src_key not in source_state:
                raise KeyError(f'Missing Phase-1 tensor {src_key}')
            src_tensor = source_state[src_key]
            if src_tensor.shape != dst_param.shape:
                raise ValueError(f'Shape mismatch for {src_key}: {tuple(src_tensor.shape)} vs {tuple(dst_param.shape)}')
            dst_param.copy_(src_tensor)

del encdec_ckpt, source_state
gc.collect()
student_params = sum(p.numel() for p in student_model.parameters())
student_trainable = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
print('Student total parameters:', f'{student_params:,}', f'({student_params/1e6:.3f}M)')
print('Student trainable parameters:', f'{student_trainable:,}')
print('Retained pretrained encoder layers:', SELECTED_ENCODER_LAYERS)
print('Reduced vocabulary:', REDUCED_VOCAB_SIZE)
assert student_params < 50_000_000, f'Student exceeds the 50M cap: {student_params:,}'

Student total parameters: 46,552,066 (46.552M)
Student trainable parameters: 46,552,066
Retained pretrained encoder layers: [0, 2, 3, 5]
Reduced vocabulary: 24361


## 9. Document windowing, batching and a forward-pass sanity check

In [ ]:
def student_n_chunks(split, i):
    L = int(S_D[split]['tail_off'][i + 1] - S_D[split]['tail_off'][i])
    return min(STUDENT_K, max(1, (L + STUDENT_BODY - 1) // STUDENT_BODY))

def build_student_chunks(split, i):
    tail = student_tail(split, i).astype(np.int64)
    body, end = [], len(tail)
    while end > 0 and len(body) < STUDENT_K:
        start = max(0, end - STUDENT_BODY)
        body.append(tail[start:end])
        end = start
    body = body[::-1] if body else [tail]
    n = len(body)
    return [(STUDENT_K - n + j, np.concatenate(([BOS_ID], chunk)).astype(np.int64)) for j, chunk in enumerate(body)]

class CJPEStudentDataset(Dataset):
    def __init__(self, split):
        self.split = split
        self.n = S_D[split]['n']
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        return (self.split, int(idx))

def collate_student(items):
    seqs, doc_index, slots, labels, t_logits = [], [], [], [], []
    split = items[0][0]
    if any(s != split for s, _ in items):
        raise ValueError('A student batch cannot mix dataset splits.')
    for b, (_, i) in enumerate(items):
        for slot, ids in build_student_chunks(split, i):
            seqs.append(ids); doc_index.append(b); slots.append(slot)
        if 'labels' in S_D[split]:
            labels.append(int(S_D[split]['labels'][i]))
        if split in TEACHER_LOGITS:
            t_logits.append(TEACHER_LOGITS[split][i])
    L = max(map(len, seqs))
    input_ids = np.full((len(seqs), L), PAD_ID, dtype=np.int64)
    lengths = np.asarray([len(x) for x in seqs], dtype=np.int64)
    for j, ids in enumerate(seqs):
        input_ids[j, :len(ids)] = ids
    batch = {
        'input_ids': torch.from_numpy(input_ids),
        'lengths': torch.from_numpy(lengths),
        'doc_index': torch.tensor(doc_index, dtype=torch.long),
        'slot': torch.tensor(slots, dtype=torch.long),
        'n_docs': len(items),
    }
    if labels:
        batch['labels'] = torch.tensor(labels, dtype=torch.long)
    if t_logits:
        batch['teacher_logits'] = torch.from_numpy(np.asarray(t_logits, dtype=np.float32))
    return batch

def move_batch(batch):
    # pack_padded_sequence requires CPU lengths. Keep them there rather than copying
    # CPU -> GPU -> CPU on every batch; all other tensors are transferred non-blockingly.
    return {k: (v if k == 'lengths' else (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)) for k, v in batch.items()}

_smoke = collate_student([('train', 0), ('train', 1)])
_smoke = move_batch(_smoke)
student_model.eval()
with torch.no_grad(), torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
    _logits = student_model(_smoke)
assert _logits.shape == (2, 2) and torch.isfinite(_logits).all()
print('Forward-pass smoke test passed | output shape:', tuple(_logits.shape))
print('Student parameter budget check passed:', f'{student_params/1e6:.2f}M < 50M')
del _smoke, _logits
gc.collect(); torch.cuda.empty_cache()

Forward-pass smoke test passed | output shape: (2, 2)
Student parameter budget check passed: 46.55M < 50M


## 10. Train the student with hard labels + teacher soft targets

`alpha=0.5` and `T=2` follow the Phase 2 distillation setup. The pretrained embedding/encoder use a smaller learning rate than the newly initialized BiLSTM and document head. The best checkpoint is selected by validation ROC-AUC; macro-F1 is reported at both threshold 0.5 and a validation-tuned threshold.

In [ ]:
def tune_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    scores = np.asarray([f1_score(y, (p >= t).astype(np.int64), average='macro', zero_division=0) for t in grid])
    j = int(scores.argmax())
    return float(grid[j]), float(scores[j])

def evaluate_student(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.inference_mode():
        for batch in loader:
            batch = move_batch(batch)
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = model(batch)
            all_logits.append(logits.float().cpu().numpy())
            all_labels.append(batch['labels'].cpu().numpy())
    logits = np.concatenate(all_logits)
    y = np.concatenate(all_labels).astype(np.int64)
    p = torch.softmax(torch.from_numpy(logits), dim=-1)[:, 1].numpy()
    auc = float(roc_auc_score(y, p))
    tuned_thr, tuned_f1 = tune_threshold(y, p)
    f1_05 = float(f1_score(y, (p >= 0.5).astype(np.int64), average='macro', zero_division=0))
    return {'logits': logits, 'labels': y, 'prob1': p, 'auc': auc,
            'threshold': tuned_thr, 'macro_f1_tuned': tuned_f1, 'macro_f1_050': f1_05}

train_loader = DataLoader(CJPEStudentDataset('train'), batch_size=BATCH_DOCS, shuffle=True,
                          num_workers=0, collate_fn=collate_student, pin_memory=True)
val_loader = DataLoader(CJPEStudentDataset('validation'), batch_size=BATCH_DOCS, shuffle=False,
                        num_workers=0, collate_fn=collate_student, pin_memory=True)

backbone_params = list(student_model.token_embedding.parameters()) + list(student_model.encoder_blocks.parameters()) + list(student_model.final_norm.parameters())
backbone_ids = {id(p) for p in backbone_params}
head_params = [p for p in student_model.parameters() if id(p) not in backbone_ids]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': PRETRAINED_LR},
    {'params': head_params, 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_updates = max(1, EPOCHS * updates_per_epoch)
warmup_updates = max(1, int(WARMUP_FRAC * total_updates))
def lr_lambda(step):
    if step < warmup_updates:
        return max(1e-3, (step + 1) / warmup_updates)
    progress = (step - warmup_updates) / max(1, total_updates - warmup_updates)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=USE_SCALER)

BEST_STUDENT_PATH = OUT_DIR / 'best_student_kd_encdec.pt'
LAST_STUDENT_PATH = OUT_DIR / 'last_student_kd_encdec.pt'
HISTORY_PATH = OUT_DIR / 'student_kd_history.json'
history = []
best_auc = -1.0
best_epoch = 0
bad_epochs = 0


def save_student_checkpoint(path, model, epoch, metrics=None):
    payload = {
        'model_state_dict': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        'student_config': {
            'vocab_size': REDUCED_VOCAB_SIZE, 'd_model': 768,
            'selected_encoder_layers_from_phase1': SELECTED_ENCODER_LAYERS,
            'retained_encoder_blocks': len(SELECTED_ENCODER_LAYERS),
            'student_chunks': STUDENT_K, 'student_body_tokens': STUDENT_BODY,
            'lstm_hidden': STUDENT_LSTM_HIDDEN, 'parameter_count': student_params,
        },
        'teacher_checkpoint': str(TEACHER_CKPT_PATH),
        'teacher_stage': 'model_state_dict_stageA',
        'encoder_decoder_checkpoint': str(ENCDEC_CKPT_PATH),
        'tokenizer_path': str(TOKENIZER_PATH),
        'distillation': {'temperature': DISTILL_TEMPERATURE, 'alpha_kd': ALPHA_KD},
        'epoch': int(epoch), 'validation_metrics': metrics or {}, 'seed': SEED,
    }
    torch.save(payload, path)

print(f'Training docs: {len(train_loader.dataset):,} | validation docs: {len(val_loader.dataset):,}')
print(f'Batches/epoch: {len(train_loader):,} | optimizer updates/epoch: {updates_per_epoch:,}')
print(f'Effective docs/update: up to {BATCH_DOCS * GRAD_ACCUM_STEPS}')

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    student_model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_values, hard_values, kd_values = [], [], []

    for step, raw_batch in enumerate(train_loader, 1):
        batch = move_batch(raw_batch)
        labels = batch['labels']
        teacher_logits = batch['teacher_logits']
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            student_logits = student_model(batch)
            hard_loss = F.cross_entropy(student_logits.float(), labels)
            teacher_soft = F.softmax(teacher_logits.float() / DISTILL_TEMPERATURE, dim=-1)
            student_log_soft = F.log_softmax(student_logits.float() / DISTILL_TEMPERATURE, dim=-1)
            kd_loss = F.kl_div(student_log_soft, teacher_soft, reduction='batchmean') * (DISTILL_TEMPERATURE ** 2)
            loss = (1.0 - ALPHA_KD) * hard_loss + ALPHA_KD * kd_loss
            scaled_loss = loss / GRAD_ACCUM_STEPS
        scaler.scale(scaled_loss).backward()
        # Keep scalar logs on GPU during the epoch; per-batch .cpu() forces synchronization.
        loss_values.append(loss.detach())
        hard_values.append(hard_loss.detach())
        kd_values.append(kd_loss.detach())

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        if step == 1 or step % 200 == 0 or step == len(train_loader):
            print(f'epoch {epoch}/{EPOCHS} | step {step:,}/{len(train_loader):,} | loss={loss_values[-1]:.4f} hard={hard_values[-1]:.4f} kd={kd_values[-1]:.4f}')

    train_means = torch.stack([
        torch.stack(loss_values).mean(),
        torch.stack(hard_values).mean(),
        torch.stack(kd_values).mean(),
    ]).float().cpu().tolist()
    val = evaluate_student(student_model, val_loader)
    row = {
        'epoch': epoch,
        'train_loss': float(train_means[0]),
        'train_hard_loss': float(train_means[1]),
        'train_kd_loss': float(train_means[2]),
        'val_auc': val['auc'],
        'val_macro_f1_threshold_050': val['macro_f1_050'],
        'val_macro_f1_tuned': val['macro_f1_tuned'],
        'val_tuned_threshold': val['threshold'],
        'minutes': round((time.time() - epoch_start) / 60, 2),
        'parameters': student_params,
    }
    history.append(row)
    improved = val['auc'] > best_auc + MIN_DELTA_AUC
    print(f"\nEpoch {epoch}: val AUC={val['auc']:.4f} | macro-F1@0.5={val['macro_f1_050']:.4f} | tuned macro-F1={val['macro_f1_tuned']:.4f} at threshold={val['threshold']:.2f} | {row['minutes']:.1f} min" + (' | BEST' if improved else ''))
    save_student_checkpoint(LAST_STUDENT_PATH, student_model, epoch, {k: v for k, v in row.items() if k != 'epoch'})
    if improved:
        best_auc = val['auc']
        best_epoch = epoch
        bad_epochs = 0
        save_student_checkpoint(BEST_STUDENT_PATH, student_model, epoch, {k: v for k, v in row.items() if k != 'epoch'})
    else:
        bad_epochs += 1
    json.dump(history, open(HISTORY_PATH, 'w', encoding='utf-8'), indent=2)
    print('Saved latest checkpoint:', LAST_STUDENT_PATH)
    if improved:
        print('Saved best checkpoint:', BEST_STUDENT_PATH)
    if bad_epochs >= PATIENCE:
        print(f'Early stopping: no validation AUC improvement for {PATIENCE} epoch(s).')
        break

print('\nTraining complete | best epoch:', best_epoch, '| best validation AUC:', round(best_auc, 4))
assert BEST_STUDENT_PATH.exists()

Training docs: 32,305 | validation docs: 994
Batches/epoch: 8,077 | optimizer updates/epoch: 4,039
Effective docs/update: up to 8
epoch 1/4 | step 1/8,077 | loss=0.9841 hard=0.9312 kd=1.0369
epoch 1/4 | step 200/8,077 | loss=0.5464 hard=0.8905 kd=0.2023
epoch 1/4 | step 400/8,077 | loss=0.6263 hard=0.7886 kd=0.4640
epoch 1/4 | step 600/8,077 | loss=0.7509 hard=0.9385 kd=0.5633
epoch 1/4 | step 800/8,077 | loss=0.4845 hard=0.6727 kd=0.2963
epoch 1/4 | step 1,000/8,077 | loss=0.7579 hard=0.6992 kd=0.8165
epoch 1/4 | step 1,200/8,077 | loss=0.4924 hard=0.5949 kd=0.3898
epoch 1/4 | step 1,400/8,077 | loss=0.3537 hard=0.6454 kd=0.0620
epoch 1/4 | step 1,600/8,077 | loss=0.5277 hard=0.5972 kd=0.4582
epoch 1/4 | step 1,800/8,077 | loss=0.6550 hard=0.9553 kd=0.3546
epoch 1/4 | step 2,000/8,077 | loss=0.4785 hard=0.4978 kd=0.4593
epoch 1/4 | step 2,200/8,077 | loss=0.4621 hard=0.5907 kd=0.3336
epoch 1/4 | step 2,400/8,077 | loss=0.3750 hard=0.4474 kd=0.3026
epoch 1/4 | step 2,600/8,077 | loss=0

## 11. Restore the best student, select a dev threshold, and write test predictions

In [ ]:
best_payload = torch.load(BEST_STUDENT_PATH, map_location='cpu', weights_only=False)
student_model.load_state_dict(best_payload['model_state_dict'], strict=True)
student_model.to(device).eval()
val = evaluate_student(student_model, val_loader)
y_dev = val['labels']
p_dev = val['prob1']
threshold = val['threshold']
val_pred = (p_dev >= threshold).astype(np.int64)
print('BEST STUDENT VALIDATION RESULTS')
print('-------------------------------')
print('checkpoint:', BEST_STUDENT_PATH)
print('best epoch:', best_payload['epoch'])
print('parameters:', f'{student_params:,}', f'({student_params/1e6:.3f}M)')
print('ROC-AUC:', f'{val["auc"]:.4f}')
print('macro-F1 @ 0.50:', f'{val["macro_f1_050"]:.4f}')
print('tuned threshold:', f'{threshold:.2f}')
print('macro-F1 @ tuned threshold:', f'{f1_score(y_dev, val_pred, average="macro", zero_division=0):.4f}')
print('macro precision:', f'{precision_score(y_dev, val_pred, average="macro", zero_division=0):.4f}')
print('macro recall:', f'{recall_score(y_dev, val_pred, average="macro", zero_division=0):.4f}')
print('confusion matrix (rows=true, columns=pred):\n', confusion_matrix(y_dev, val_pred))


test_loader = DataLoader(CJPEStudentDataset('test'), batch_size=BATCH_DOCS, shuffle=False,
                         num_workers=0, collate_fn=collate_student, pin_memory=True)
def predict_student(loader):
    student_model.eval()
    outs = []
    with torch.inference_mode():
        for batch in loader:
            batch = move_batch(batch)
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = student_model(batch)
            outs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(outs)

p_test = predict_student(test_loader)
test_ids = [str(x) for x in S_D['test']['ids'].tolist()]
test_pred = (p_test >= threshold).astype(np.int64)
SUBMISSION_PATH = OUT_DIR / 'submission_kd_encdec_student.json'
with open(SUBMISSION_PATH, 'w', encoding='utf-8') as f:
    json.dump([{'id': cid, 'pred': int(pred)} for cid, pred in zip(test_ids, test_pred)], f, indent=2)
np.save(OUT_DIR / 'student_test_probabilities.npy', p_test)

assert len(test_ids) == len(test_pred) == 1517
assert len(set(test_ids)) == len(test_ids)
assert all(int(x) in (0, 1) for x in test_pred)
print('\nTEST OUTPUT')
print('------------')
print('test documents:', len(test_ids))
print('predicted class counts:', {c: int((test_pred == c).sum()) for c in (0, 1)})
print('submission:', SUBMISSION_PATH)

BEST STUDENT VALIDATION RESULTS
-------------------------------
checkpoint: /content/drive/MyDrive/phase4_kd_encdec_student/best_student_kd_encdec.pt
best epoch: 2
parameters: 46,552,066 (46.552M)
ROC-AUC: 0.8124
macro-F1 @ 0.50: 0.7027
tuned threshold: 0.33
macro-F1 @ tuned threshold: 0.7374
macro precision: 0.7374
macro recall: 0.7374
confusion matrix (rows=true, columns=pred):
 [[364 133]
 [128 369]]

TEST OUTPUT
------------
test documents: 1517
predicted class counts: {0: 693, 1: 824}
submission: /content/drive/MyDrive/phase4_kd_encdec_student/submission_kd_encdec_student.json


## 12. Save experiment summary and list output files

In [ ]:
summary = {
    'experiment': 'Phase 4 CJPE knowledge distillation: Encoder+FT teacher -> compressed Phase-1 encoder-derived student',
    'teacher_checkpoint': str(TEACHER_CKPT_PATH),
    'teacher_stage_used': 'model_state_dict_stageA (train-only weights)',
    'student_initialization_checkpoint': str(ENCDEC_CKPT_PATH),
    'student_tokenizer_source': str(TOKENIZER_PATH),
    'student_vocab_size': REDUCED_VOCAB_SIZE,
    'student_selected_encoder_layers': SELECTED_ENCODER_LAYERS,
    'student_parameter_count': int(student_params),
    'student_parameter_count_millions': float(student_params / 1e6),
    'under_50M_assertion': bool(student_params < 50_000_000),
    'student_chunks': STUDENT_K,
    'student_body_tokens': STUDENT_BODY,
    'student_lstm_hidden': STUDENT_LSTM_HIDDEN,
    'distillation_temperature': DISTILL_TEMPERATURE,
    'alpha_kd': ALPHA_KD,
    'best_epoch': int(best_payload['epoch']),
    'validation_auc': float(val['auc']),
    'validation_macro_f1_at_050': float(val['macro_f1_050']),
    'validation_tuned_threshold': float(threshold),
    'validation_macro_f1_tuned': float(f1_score(y_dev, val_pred, average='macro', zero_division=0)),
    'predicted_test_class_counts': {str(c): int((test_pred == c).sum()) for c in (0, 1)},
    'submission_path': str(SUBMISSION_PATH),
}
SUMMARY_PATH = OUT_DIR / 'kd_encdec_student_summary.json'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print('\nOUTPUT FILES')
for path in sorted(OUT_DIR.glob('*')):
    if path.is_file():
        print(f'{path.name:<44} {path.stat().st_size / 1024**2:>8.2f} MB')
assert student_params < 50_000_000

{
  "experiment": "Phase 4 CJPE knowledge distillation: Encoder+FT teacher -> compressed Phase-1 encoder-derived student",
  "teacher_checkpoint": "/content/drive/MyDrive/phase4_exp1_v2_encoder_full_ft/exp1_v2_encoder_full_ft_model.pt",
  "teacher_stage_used": "model_state_dict_stageA (train-only weights)",
  "student_initialization_checkpoint": "/content/drive/MyDrive/best_model_encdec.pt",
  "student_tokenizer_source": "/content/drive/MyDrive/legal_bpe_a1_50k_final.json",
  "student_vocab_size": 24361,
  "student_selected_encoder_layers": [
    0,
    2,
    3,
    5
  ],
  "student_parameter_count": 46552066,
  "student_parameter_count_millions": 46.552066,
  "under_50M_assertion": true,
  "student_chunks": 4,
  "student_body_tokens": 703,
  "student_lstm_hidden": 192,
  "distillation_temperature": 2.0,
  "alpha_kd": 0.5,
  "best_epoch": 2,
  "validation_auc": 0.8123712091462254,
  "validation_macro_f1_at_050": 0.7027274067521745,
  "validation_tuned_threshold": 0.32999999999999996,